# League of Legends: Classification EDA, Logistic Regression, and Decision Trees

Owner: Isaac. This notebook builds on `01_data_exploration.ipynb`, which creates the minute-14 dataset, the engineered features, and the match-level train/test split.

**Run `01_data_exploration.ipynb` first** so `data/processed/` contains `engineered_data.csv`, `engineered_feature_columns.csv`, and `split_ids.csv`.

Rules followed here:
- Only **training** matches are used. The test matches stay untouched until the team's final evaluation.
- Cross-validation uses `StratifiedGroupKFold` on `gameId`, so all snapshots of a match stay in the same fold.
- Scaling happens inside a pipeline, so it is fit on each training fold only.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

SEED = 42

# Support notebooks launched from the repository root or notebooks/.
root = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
processed_dir = root / "data" / "processed"
report_dir = root / "reports"
figure_dir = report_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

required = ["engineered_data.csv", "engineered_feature_columns.csv", "split_ids.csv"]
missing = [f for f in required if not (processed_dir / f).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {processed_dir}. Run 01_data_exploration.ipynb first."
    )

In [ ]:
data = pd.read_csv(processed_dir / "engineered_data.csv")
feature_cols = pd.read_csv(processed_dir / "engineered_feature_columns.csv")["feature"].tolist()
split_ids = pd.read_csv(processed_dir / "split_ids.csv")
raw = pd.read_csv(root / "data" / "raw" / "lol_ranked_games.csv")

train_ids = set(split_ids.loc[split_ids["split"].eq("train"), "gameId"])
train = data.loc[data["gameId"].isin(train_ids)].reset_index(drop=True)

# One row per match at minute 14
assert train["gameId"].is_unique

print(f"Minute-14 training matches: {len(train):,} of {len(data):,}")
print(f"Blue win rate (train): {train['hasWon'].mean():.3f}")
print(f"Candidate features from notebook 01: {len(feature_cols)}")

## 1. Exploration for classification

### 1.1 What do `isFirstTower` and `isFirstBlood` actually mean?

The names suggest "blue took first tower" and "blue got first blood". We check this against the turret and kill counters.

In [ ]:
lanes = ["Top", "Mid", "Bot"]
tiers = ["Outer", "Inner", "Base", "Nexus"]
blue_turrets = train[[f"destroyed{l}{t}Turret" for l in lanes for t in tiers]].sum(axis=1)
red_turrets = train[[f"lost{l}{t}Turret" for l in lanes for t in tiers]].sum(axis=1)

print("isFirstTower vs. turrets destroyed by each team:")
display(pd.crosstab(
    train["isFirstTower"],
    [blue_turrets.gt(0).rename("blue took a turret"),
     red_turrets.gt(0).rename("red took a turret")],
))

print("isFirstBlood vs. who has kills so far:")
display(pd.crosstab(
    train["isFirstBlood"],
    [train["kills"].gt(0).rename("blue has a kill"),
     train["deaths"].gt(0).rename("red has a kill")],
))

**Findings**

- `isFirstTower` is 1 exactly when blue has destroyed at least one turret, even when red also has one. It behaves like "blue has taken a turret", not strictly "blue took the *first* turret".
- `isFirstBlood` is 1 whenever *any* kill has happened, including matches where only red has kills. It records whether first blood has happened, not which team got it. At minute 14 it is almost always 1, so it carries no information about the winner. **We drop it from the models.**

In [ ]:
constant_cols = [c for c in feature_cols if train[c].nunique() <= 1]
drop_cols = constant_cols + ["isFirstBlood"]
model_cols = [c for c in feature_cols if c not in drop_cols]

print("Constant at minute 14 (dropped):", constant_cols)
print("Model features:", len(model_cols))

### 1.2 Which minute-14 advantages go with winning?

Notebook 01 already covers gold position and dragons. Here: turrets, Rift Herald, and kill difference. Groups with fewer than 50 matches are hidden.

In [ ]:
panels = {
    "Outer turret difference (blue - red)": train["outerTurretDiff"],
    "Rift Herald difference (blue - red)": train["killedRiftHerald"] - train["lostRiftHerald"],
    "Blue has destroyed a turret": train["isFirstTower"].map({0: "No", 1: "Yes"}),
    "Kill - death difference (clipped at ±6)": train["killDeathDiff"].clip(-6, 6),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, (title, values) in zip(axes.flat, panels.items()):
    table = train.groupby(values.rename("value"))["hasWon"].agg(matches="size", win_rate="mean")
    table = table[table["matches"] >= 50]
    ax.bar(table.index.astype(str), table["win_rate"], color="#2a9d8f")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    for x, (n, rate) in enumerate(zip(table["matches"], table["win_rate"])):
        ax.text(x, rate + 0.02, f"n={n:,}", ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Blue win rate")

fig.suptitle("Blue win rate by minute-14 advantage (training matches)")
fig.tight_layout()
fig.savefig(figure_dir / "objective_win_rates.png", dpi=200, bbox_inches="tight")
plt.show()

### 1.3 How predictive is gold over time, and how often do teams come back?

For each prediction minute, we keep training matches still in progress at that minute (same duration rule as notebook 01) and plot blue's win rate against gold difference.

In [ ]:
def frame_snapshot(frame):
    """One row per training match still in progress at the given minute."""
    rows = raw.loc[raw["frame"].eq(frame) & raw["gameDuration"].gt(frame * 60_000)]
    return rows.loc[rows["gameId"].isin(train_ids)].reset_index(drop=True)


frames = [10, 14, 18, 20]
gold_bins = np.arange(-8000, 8001, 1000)

curves = []
for frame in frames:
    snap = frame_snapshot(frame)
    binned = pd.cut(snap["goldDiff"].clip(-7999, 7999), gold_bins)
    table = (
        snap.groupby(binned, observed=True)["hasWon"]
        .agg(matches="size", win_rate="mean")
        .reset_index()
    )
    table["gold_k"] = [interval.mid / 1000 for interval in table["goldDiff"]]
    table["minute"] = frame
    curves.append(table[table["matches"] >= 30])
curves = pd.concat(curves, ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=curves, x="gold_k", y="win_rate", hue="minute", marker="o", palette="viridis", ax=ax)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.axvline(0, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("Blue gold difference (thousands; outer bins include tails)")
ax.set_ylabel("Blue win rate")
ax.set_title("The same gold lead matters less as the game goes on")
fig.tight_layout()
fig.savefig(figure_dir / "gold_win_curve_by_minute.png", dpi=200, bbox_inches="tight")
plt.show()

display(curves.pivot(index="gold_k", columns="minute", values="win_rate").round(2))

**Comebacks at minute 14.** Among training matches where blue is behind in gold, how often does blue still win, by deficit size and by whether blue closed the gap between minutes 10 and 14?

In [ ]:
behind = train.loc[train["goldDiff"] < 0].copy()
behind["deficit"] = pd.cut(
    -behind["goldDiff"],
    [0, 1000, 2000, 3000, 5000, np.inf],
    labels=["0-1k", "1-2k", "2-3k", "3-5k", "5k+"],
)
behind["trend 10->14"] = np.where(
    behind["goldChange10to14"] > 0, "Closing the gap", "Not closing"
)

comeback = (
    behind.groupby(["deficit", "trend 10->14"], observed=True)["hasWon"]
    .agg(matches="size", win_rate="mean")
)
display(comeback.unstack().round(3))

# Hide groups too small to give a reliable rate (e.g. 5k+ behind but closing the gap)
plot_comeback = comeback.reset_index()
plot_comeback = plot_comeback[plot_comeback["matches"] >= 30]

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_comeback, x="deficit", y="win_rate", hue="trend 10->14",
    palette=["#2a9d8f", "#d97757"], ax=ax,
)
ax.set_xlabel("Blue gold deficit at minute 14 (groups with < 30 matches hidden)")
ax.set_ylabel("Blue win rate (comeback rate)")
ax.set_title("Comeback rate by deficit size and recent momentum")
fig.tight_layout()
fig.savefig(figure_dir / "comeback_rates.png", dpi=200, bbox_inches="tight")
plt.show()

### 1.4 Correlated predictors

Strongly correlated features make logistic-regression coefficients hard to interpret: the model can split credit between them arbitrarily. `killDeathDiff` is exactly `kills - deaths`, and `dragonDiff` and `outerTurretDiff` are sums of raw counters.

In [ ]:
display(train[["goldDiff", "expDiff", "champLevelDiff", "killDeathDiff", "kills", "deaths"]].corr().round(2))

## 2. Evaluation setup and baselines

- **5-fold `StratifiedGroupKFold`** on training matches, grouped by `gameId`.
- **Metrics:** accuracy (classification), ROC AUC (ranking), and log loss / Brier score (quality of win probabilities; lower is better).
- **Baselines:** always predict the majority class, and a *gold-lead rule* that predicts the win rate of "ahead" or "behind" teams observed in the training fold.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "log_loss": "neg_log_loss",
    "brier": "neg_brier_score",
}
lower_is_better = {"log_loss", "brier"}


def fold_scores(raw_scores):
    """Turn sklearn's negated losses back into positive values."""
    return {m: -v if m in lower_is_better else v for m, v in raw_scores.items()}


def cv_scores(model, data, cols):
    result = cross_validate(
        model, data[cols], data["hasWon"], groups=data["gameId"], cv=cv, scoring=scoring
    )
    return fold_scores({m: result[f"test_{m}"] for m in scoring})


def tune(model, grid, data, cols):
    """Grid search with grouped CV; the best setting is chosen by log loss."""
    search = GridSearchCV(
        model, grid, cv=cv, scoring=scoring, refit="log_loss",
        n_jobs=-1, return_train_score=True,
    )
    search.fit(data[cols], data["hasWon"], groups=data["gameId"])
    i = search.best_index_
    n_splits = cv.get_n_splits()
    scores = {
        m: np.array([search.cv_results_[f"split{k}_test_{m}"][i] for k in range(n_splits)])
        for m in scoring
    }
    return search, fold_scores(scores)


def summarize(model_name, feature_set, scores, params=None):
    row = {"model": model_name, "features": feature_set}
    row.update({m: v.mean() for m, v in scores.items()})
    row["accuracy_sd"] = scores["accuracy"].std()
    row["params"] = params or {}
    return row


class GoldLeadRule(ClassifierMixin, BaseEstimator):
    """Predict the training win rate of teams ahead (goldDiff > 0) or not ahead."""

    def fit(self, X, y):
        ahead = np.asarray(X["goldDiff"]) > 0
        y = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.p_ahead_ = y[ahead].mean()
        self.p_not_ahead_ = y[~ahead].mean()
        return self

    def predict_proba(self, X):
        p = np.where(np.asarray(X["goldDiff"]) > 0, self.p_ahead_, self.p_not_ahead_)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


results = [
    summarize("Majority class", "none", cv_scores(DummyClassifier(strategy="prior"), train, ["goldDiff"])),
    summarize("Gold-lead rule", "sign of goldDiff", cv_scores(GoldLeadRule(), train, ["goldDiff"])),
]
metric_cols = ["accuracy", "accuracy_sd", "roc_auc", "log_loss", "brier"]
display(pd.DataFrame(results)[["model", "features"] + metric_cols].round(4))

## 3. Logistic regression

We compare feature sets to answer the proposal's question: **do objectives and changes in advantage add information beyond the resource lead?**

| Feature set | Contents |
| --- | --- |
| Gold only | `goldDiff` |
| Resources | gold, experience, and champion-level differences |
| Resources + changes | + gold/experience change from minute 10 to 14 |
| Resources + objectives | + kill difference, dragons, outer turrets, turret flag, Rift Herald |
| Compact | resources + changes + objectives |
| All features | every remaining raw and engineered feature |

The regularization strength `C` is tuned per feature set.

In [ ]:
resources = ["goldDiff", "expDiff", "champLevelDiff"]
changes = ["goldChange10to14", "expChange10to14"]
objectives = [
    "killDeathDiff", "dragonDiff", "outerTurretDiff",
    "isFirstTower", "killedRiftHerald", "lostRiftHerald",
]
compact = resources + changes + objectives

feature_sets = {
    "Gold only": ["goldDiff"],
    "Resources": resources,
    "Resources + changes": resources + changes,
    "Resources + objectives": resources + objectives,
    "Compact": compact,
    "All features": model_cols,
}

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
lr_grid = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]}

lr_searches = {}
for set_name, cols in feature_sets.items():
    search, scores = tune(lr, lr_grid, train, cols)
    lr_searches[set_name] = search
    results.append(summarize("Logistic regression", set_name, scores, search.best_params_))

lr_table = pd.DataFrame([r for r in results if r["model"] == "Logistic regression"])
display(lr_table[["features"] + metric_cols + ["params"]].round(4))

**Coefficients of the compact model.** Features are standardized, so each coefficient is the change in log-odds of a blue win for a one-standard-deviation increase in that feature, holding the others fixed. Correlated features (see 1.4) share credit, so read the coefficients together rather than one at a time.

In [ ]:
compact_lr = lr_searches["Compact"].best_estimator_
coefs = pd.DataFrame({
    "coefficient": compact_lr[-1].coef_[0],
    "odds_ratio_per_sd": np.exp(compact_lr[-1].coef_[0]),
}, index=compact).sort_values("coefficient")

display(coefs.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(coefs["coefficient"] > 0, "#2a9d8f", "#d97757")
ax.barh(coefs.index, coefs["coefficient"], color=colors)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Standardized coefficient (log-odds per 1 SD)")
ax.set_title("Logistic regression (compact features, minute 14)")
fig.tight_layout()
fig.savefig(figure_dir / "logistic_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Decision tree

We tune `max_depth` and `min_samples_leaf` on the same feature sets. Trees don't need scaling.

In [ ]:
dt = DecisionTreeClassifier(random_state=SEED)
dt_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": [1, 20, 50, 100, 200],
}

dt_searches = {}
for set_name, cols in feature_sets.items():
    search, scores = tune(dt, dt_grid, train, cols)
    dt_searches[set_name] = search
    results.append(summarize("Decision tree", set_name, scores, search.best_params_))

dt_table = pd.DataFrame([r for r in results if r["model"] == "Decision tree"])
display(dt_table[["features"] + metric_cols + ["params"]].round(4))

**Effect of depth alone.** With `min_samples_leaf=1`, deeper trees fit the training folds better and the validation folds worse: the overfitting the tuning protects against.

In [ ]:
cv_table = pd.DataFrame(dt_searches["Compact"].cv_results_)
depth_curve = cv_table.loc[cv_table["param_min_samples_leaf"].eq(1)].copy()
depth_curve["depth"] = depth_curve["param_max_depth"].map(lambda d: "None" if d is None or pd.isna(d) else str(int(d)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(depth_curve["depth"], depth_curve["mean_train_accuracy"], marker="o", label="training folds")
axes[0].plot(depth_curve["depth"], depth_curve["mean_test_accuracy"], marker="o", label="validation folds")
axes[0].set_ylabel("Accuracy")
axes[1].plot(depth_curve["depth"], -depth_curve["mean_test_log_loss"], marker="o", color="#d97757")
axes[1].set_yscale("log")
axes[1].set_ylabel("Validation log loss, log scale (lower is better)")
for ax in axes:
    ax.set_xlabel("max_depth")
axes[0].legend()
fig.suptitle("Decision tree depth vs. performance (compact features, min_samples_leaf=1)")
fig.tight_layout()
fig.savefig(figure_dir / "tree_depth_curve.png", dpi=200, bbox_inches="tight")
plt.show()

**A readable tree for the presentation.** A depth-3 tree on the compact features. Its cross-validated scores are shown so we know how much accuracy we give up for readability.

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=SEED)
small_scores = cv_scores(small_tree, train, compact)
results.append(summarize("Decision tree (depth 3, for display)", "Compact", small_scores,
                         {"max_depth": 3, "min_samples_leaf": 100}))
print({m: round(v.mean(), 4) for m, v in small_scores.items()})

small_tree.fit(train[compact], train["hasWon"])
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    small_tree, feature_names=compact, class_names=["Blue loss", "Blue win"],
    filled=True, rounded=True, proportion=True, impurity=False, fontsize=9, ax=ax,
)
ax.set_title("Depth-3 decision tree at minute 14 (training matches)")
fig.savefig(figure_dir / "decision_tree_depth3.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
best_tree = dt_searches["All features"].best_estimator_
importance = (
    pd.Series(best_tree.feature_importances_, index=model_cols)
    .sort_values(ascending=False)
)
top = importance[importance > 0].head(15).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index, top.values, color="#2a9d8f")
ax.set_xlabel("Impurity-based importance")
ax.set_title(f"Tuned decision tree, all features ({dt_searches['All features'].best_params_})")
fig.tight_layout()
fig.savefig(figure_dir / "tree_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Minute-14 comparison

In [ ]:
summary = pd.DataFrame(results)
display(summary[["model", "features"] + metric_cols].round(4))

summary.assign(params=summary["params"].astype(str)).to_csv(
    report_dir / "lr_dt_cv_results.csv", index=False
)

plot_data = summary.loc[summary["model"].isin(["Logistic regression", "Decision tree"])]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, metric in zip(axes, ["accuracy", "log_loss"]):
    sns.barplot(data=plot_data, y="features", x=metric, hue="model",
                palette=["#2a9d8f", "#d97757"], ax=ax)
    for _, base in summary.loc[summary["model"].eq("Gold-lead rule")].iterrows():
        ax.axvline(base[metric], color="gray", linestyle="--", linewidth=1, label="Gold-lead rule")
    ax.set_xlim(0.4 if metric == "log_loss" else 0.6, None)
    ax.set_ylabel("")
handles, labels = axes[1].get_legend_handles_labels()
for ax in axes:
    ax.get_legend().remove()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
axes[0].set_title("CV accuracy (higher is better)")
axes[1].set_title("CV log loss (lower is better)")
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(figure_dir / "minute14_model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Prediction time: minutes 10, 14, 18, 20

At each minute we use matches still in progress and the features available at that time. To keep the feature set identical across minutes, we leave out the 10-to-14 change features (they don't exist at minute 10) and recompute `dragonDiff`, `killDeathDiff`, and `outerTurretDiff` from that minute's counters. Because the split is by match, each minute uses only training matches.

In [ ]:
def add_static_features(snap):
    """Same definitions as notebook 01, computed at any minute."""
    out = snap.copy()
    drakes = ["Fire", "Water", "Air", "Earth", "Elder"]
    out["dragonDiff"] = (
        out[[f"killed{d}Drake" for d in drakes]].sum(axis=1)
        - out[[f"lost{d}Drake" for d in drakes]].sum(axis=1)
    )
    out["killDeathDiff"] = out["kills"] - out["deaths"]
    out["outerTurretDiff"] = (
        out[[f"destroyed{x}OuterTurret" for x in lanes]].sum(axis=1)
        - out[[f"lost{x}OuterTurret" for x in lanes]].sum(axis=1)
    )
    return out


frame_cols = [c for c in feature_cols if c not in changes + ["isFirstBlood"]]

frame_rows = []
for frame in frames:
    snap = add_static_features(frame_snapshot(frame))
    n = len(snap)
    for name, scores in [
        ("Gold-lead rule", cv_scores(GoldLeadRule(), snap, ["goldDiff"])),
        ("Logistic regression", tune(lr, lr_grid, snap, frame_cols)[1]),
        ("Decision tree", tune(dt, dt_grid, snap, frame_cols)[1]),
    ]:
        row = summarize(name, "All features (no change features)", scores)
        row.update({"minute": frame, "train_matches": n})
        frame_rows.append(row)

by_minute = pd.DataFrame(frame_rows)
by_minute.drop(columns="params").to_csv(report_dir / "frame_comparison.csv", index=False)

print("Eligible training matches per minute:")
display(by_minute.drop_duplicates("minute").set_index("minute")["train_matches"])
display(by_minute.pivot(index="minute", columns="model", values=["accuracy", "roc_auc", "log_loss"]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, label in [
    (axes[0], "accuracy", "CV accuracy"),
    (axes[1], "log_loss", "CV log loss (lower is better)"),
]:
    sns.lineplot(data=by_minute, x="minute", y=metric, hue="model", marker="o",
                 palette=["gray", "#2a9d8f", "#d97757"], ax=ax)
    ax.set_xticks(frames)
    ax.set_ylabel(label)
fig.suptitle("Prediction quality by prediction minute (training matches, 5-fold grouped CV)")
fig.tight_layout()
fig.savefig(figure_dir / "accuracy_by_minute.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Findings (5-fold grouped CV on training matches; test set not used)

**Exploration**
- `isFirstBlood` records whether any kill has happened, not who got it, so it was dropped. `isFirstTower` behaves like "blue has destroyed a turret".
- Every advantage tracks winning: outer-turret difference is the sharpest (about 1% win rate at -3, about 95% at +3), followed by kill difference and Rift Herald.
- The same gold lead matters less as the game goes on: blue wins about 81% of games when ahead by 2-3k at minute 10, but about 70% at minute 20.
- Comebacks: when blue is 5k+ behind at minute 14 and not closing the gap, it wins about 4% of the time. Closing the gap between minutes 10 and 14 raises the comeback rate at every deficit size.
- Gold, experience, level, and kill differences are highly correlated (0.83-0.95), so coefficients should be read together.

**Minute-14 models**
- Baselines: majority class 50.3% accuracy; gold-lead rule 76.2%.
- Logistic regression on gold alone matches the gold-lead rule in accuracy (76.2%) but gives much better probabilities (log loss 0.485 vs. 0.549).
- **Objectives add a small but consistent gain beyond resources:** logistic regression improves from 76.4% / log loss 0.479 (resources) to 77.1% / 0.471 (resources + objectives). The fold-to-fold standard deviation is about 0.9 percentage points, so accuracy gaps this small should not be over-claimed; log loss and AUC move the same way.
- **The 10-to-14 change features add nothing** once the current gold and experience differences are known.
- Using all 50 features does not beat the compact 11-feature set.
- In the compact model, `goldDiff` dominates, followed by `expDiff` and `dragonDiff`. The small negative coefficients on `champLevelDiff` and `isFirstTower` come from correlated features sharing credit, not from those advantages hurting blue.
- Decision trees are slightly worse than logistic regression on every feature set (best 76.5%). Tuning settles on depth 4-5 with at least 200 matches per leaf. Unrestricted trees overfit badly: 100% training accuracy, 68% validation accuracy.
- The depth-3 tree uses only `goldDiff` and `expDiff` and still reaches 76.1%, which makes it a good presentation figure.

**Prediction minute**
- Accuracy rises from about 72% at minute 10 to 77% at 14, 80% at 18, and 81% at 20. Logistic regression is the best of the three at every minute.
- Eligible training matches fall from 19,912 at minute 10 to 17,738 at minute 20, because games that ended earlier drop out.

## 8. Next steps

- **Final test evaluation:** after the team freezes the feature sets and hyperparameters, refit the chosen models on all training matches and evaluate them **once** on the test matches in `split_ids.csv` (coordinate with Yejune, who handles the random forest and the final evaluation).
- Add per-minute change features (for example, gold change over the previous 4 minutes) for minutes 14, 18, and 20.
- Error analysis: look at confidently wrong predictions, such as large minute-14 gold leads that still lost.